# U-Net (ResNet34, ImageNet pretrained) - unmasked

Segmentation training on IDRiD tiles. Model: U-Net with `resnet34` encoder, weights='imagenet'. Arm: **unmasked**. Loss: Dice + weighted BCE. Four classes (Microaneurysms, Haemorrhages, Hard Exudates, Optic Disc).

**Attach:** 15_idrid_preprocessing (the .npz tiles). GPU on. Saves `unet_pretrained_unmasked.pth` and `unet_pretrained_unmasked_testpreds.npz` for the metrics notebook.

In [1]:
!pip install segmentation-models-pytorch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.4 MB/s eta 0:00:00


In [2]:
import numpy as np, torch, torch.nn as nn, os
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(device)
CLASSES=['Microaneurysms','Haemorrhages','Hard Exudates','Optic Disc']
ARM='unmasked'
MODEL_SLUG='unet_pretrained'
SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)

cuda


In [3]:
# locate the tile npz files
def find(name):
    for r,_,fs in os.walk('/kaggle/input'):
        if name in fs: return os.path.join(r,name)
    raise FileNotFoundError(name)
tr=np.load(find(f'idrid_train_{ARM}.npz'))
te=np.load(find(f'idrid_test_{ARM}.npz'))
Xtr,Ytr=tr['X'],tr['Y']; Xte,Yte=te['X'],te['Y']
print('train',Xtr.shape,Ytr.shape,'| test',Xte.shape,Yte.shape)

train (1792, 512, 512, 3) (1792, 512, 512, 4) | test (1011, 512, 512, 3) (1011, 512, 512, 4)


In [4]:
IMAGENET_MEAN=np.array([0.485,0.456,0.406]); IMAGENET_STD=np.array([0.229,0.224,0.225])
class SegDS(Dataset):
    def __init__(self, X, Y, aug=False):
        self.X=X; self.Y=Y; self.aug=aug
    def __len__(self): return len(self.X)
    def __getitem__(self,i):
        img=self.X[i].astype(np.float32)/255.0
        img=(img-IMAGENET_MEAN)/IMAGENET_STD
        msk=self.Y[i].astype(np.float32)
        if self.aug:
            if np.random.rand()<0.5: img=img[:,::-1].copy(); msk=msk[:,::-1].copy()
            if np.random.rand()<0.5: img=img[::-1].copy(); msk=msk[::-1].copy()
        img=torch.from_numpy(img).permute(2,0,1).float()
        msk=torch.from_numpy(msk).permute(2,0,1).float()
        return img,msk
tr_loader=DataLoader(SegDS(Xtr,Ytr,aug=True),batch_size=8,shuffle=True,num_workers=2)
te_loader=DataLoader(SegDS(Xte,Yte,aug=False),batch_size=8,shuffle=False,num_workers=2)

In [5]:
model=smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=len(CLASSES))
model=model.to(device)
# Dice + weighted BCE
dice_loss=smp.losses.DiceLoss(mode='multilabel')
# pos_weight upweights foreground; tuned high for the sparse classes
pos_weight=torch.tensor([10.,5.,5.,3.]).view(1,4,1,1).to(device)
bce=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
def criterion(logits,target): return dice_loss(logits,target)+bce(logits,target)
opt=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-5)
EPOCHS=25
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [6]:
for epoch in range(EPOCHS):
    model.train(); tot=0
    for img,msk in tr_loader:
        img,msk=img.to(device),msk.to(device)
        opt.zero_grad(); out=model(img); loss=criterion(out,msk)
        loss.backward(); opt.step(); tot+=loss.item()
    sched.step()
    if epoch%5==0 or epoch==EPOCHS-1:
        print(f'epoch {epoch+1}/{EPOCHS} loss {tot/len(tr_loader):.4f}')
print('training done')

epoch 1/25 loss 1.3665
epoch 6/25 loss 0.6836
epoch 11/25 loss 0.4479
epoch 16/25 loss 0.3859
epoch 21/25 loss 0.3414
epoch 25/25 loss 0.3395
training done


In [7]:
# save weights + test-set predictions (probabilities) for the metrics notebook
model.eval(); probs=[]; gts=[]
with torch.no_grad():
    for img,msk in te_loader:
        out=torch.sigmoid(model(img.to(device))).cpu().numpy()
        probs.append(out); gts.append(msk.numpy())
probs=np.concatenate(probs); gts=np.concatenate(gts)
OUT='/kaggle/working'
torch.save(model.state_dict(), f'{OUT}/{MODEL_SLUG}_{ARM}.pth')
# store probs as float16 to save space
np.savez_compressed(f'{OUT}/{MODEL_SLUG}_{ARM}_testpreds.npz', probs=probs.astype(np.float16), gts=gts.astype(np.uint8))
print('saved', f'{MODEL_SLUG}_{ARM}.pth', 'and testpreds')
# quick per-class Dice sanity at threshold 0.5
p=(probs>0.5).astype(np.uint8)
for ci,c in enumerate(CLASSES):
    inter=(p[:,ci]*gts[:,ci]).sum(); denom=p[:,ci].sum()+gts[:,ci].sum()
    dice=2*inter/denom if denom>0 else 0
    print(f'  {c:<16} Dice={dice:.4f}')

saved unet_pretrained_unmasked.pth and testpreds
  Microaneurysms   Dice=0.5157
  Haemorrhages     Dice=0.5675
  Hard Exudates    Dice=0.8109
  Optic Disc       Dice=0.9464
